# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding A — "High Search Volume = More Traffic" (Myth #2, REVERSED)

**Where the label comes from.** The reversal compares two things: the stored keyword-volume
benchmark (`search_volume`, a keyword-tool estimate) and measured page impressions from GSC. The
headline numbers are the raw SV <-> impressions correlation (r = 0.0083; log-scaled −0.0419) and
"82.89% of pages with non-zero volume show impressions above the stored search-volume figure."

**Does the validation design carry the claim?** Partly. The reversal is directionally consistent
(low-volume buckets are the easiest to outperform, the 10K+ bucket the hardest), but the numbers come
from the local active-content feature-vector subset — pages with impressions > 0 AND sessions > 0.
That filter is survivorship: pages that got zero traffic are excluded, so the correlation and the
82.89% share are computed only on pages that already won visibility. No p-values or confidence
intervals are reported anywhere (stated in Methodology), and comparing an impressions count against a
stored monthly volume estimate mixes units unless the scales are reconciled.

**Why it matters for my slice:** my own w05 model leaned on `search_volume` (top permutation
importance) yet the ML-07 signal audit measured only weak point-biserial correlations with `at_risk`
(|r| ≈ 0.04–0.10). The paper's reversal and my slice agree: search volume is a weak traffic forecast,
even when it is the least-weak feature I have.

### Finding B — "The Freshness Multiplier" (Finding #4)

**Where the label comes from.** The growth-to-decline ratio is computed from "trend direction,"
defined in the paper as 30-day vs previous-30-day impression change. Refreshing 365+ day content is
reported as "57x more impressions (71 --> 4039)" and the 361+ freshness bucket as a 283:1 ratio.

**Does the validation design carry the claim?** The 283:1 headline rests on a bucket with a single
declining page — the paper itself flags it ("361+ spikes to 283:1 only because the sample is tiny and
there is just 1 declining page in that bucket"). No p-values or confidence intervals are reported
anywhere (stated in Methodology), and no base rate is printed next to the ratio, so a 283:1 ratio
against n=2 is fragile. The 57x figure is a within-bucket comparison, still observational — pages
were not randomly assigned to refresh.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*


In [2]:
import importlib
if any(importlib.util.find_spec(m) is None for m in ("duckdb", "huggingface_hub", "sklearn", "pandas", "numpy")):
    get_ipython().run_line_magic("pip", "-q install duckdb huggingface_hub scikit-learn pandas numpy")

import os
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('hf_key')
except Exception:
    HF_TOKEN = os.getenv('hf_key')
if not HF_TOKEN:
    raise RuntimeError('hf_key not found: set the Colab secret or the hf_key env var')

import duckdb
import numpy as np
import pandas as pd
from pathlib import Path

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':      f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':      f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':       f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
}

for name in ('dim_clients', 'dim_content', 'fact_daily_sample'):
    n = con.sql(f'SELECT COUNT(*) FROM {TABLES[name]}').fetchone()[0]
    print(f'{name:20} {n:>12,} rows')

repo = Path.cwd()
while not (repo / "work").exists() and repo != repo.parent:
    repo = repo.parent

cache_f = repo / "work" / "outputs" / "w05_feature_vector.parquet"
if cache_f.exists():
    df = pd.read_parquet(cache_f)
    print("loaded cached feature vector:", cache_f)
else:
    df = con.sql(f"""
        WITH bounds AS (SELECT DATE '2025-08-31' AS end_d),
        windowed AS (
            SELECT
                q.client_hash_id,
                q.content_hash_id,
                ANY_VALUE(d.main_intent)    AS main_intent,
                ANY_VALUE(d.content_type)   AS content_type,
                ANY_VALUE(d.search_volume)  AS search_volume,
                SUM(CASE WHEN q.report_date > b.end_d - INTERVAL 60 DAY AND q.report_date <= b.end_d - INTERVAL 30 DAY
                         THEN q.gsc_impressions ELSE 0 END) AS impressions_prev30d,
                SUM(CASE WHEN q.report_date > b.end_d - INTERVAL 30 DAY THEN q.gsc_impressions ELSE 0 END) AS impressions_last30d,
                SUM(CASE WHEN q.report_date > b.end_d - INTERVAL 30 DAY THEN q.gsc_clicks ELSE 0 END) AS clicks_30d,
                AVG(CASE WHEN q.report_date > b.end_d - INTERVAL 30 DAY THEN q.gsc_avg_position END) AS avg_position_30d,
                100.0 * COALESCE(CAST(SUM(CASE WHEN q.report_date > b.end_d - INTERVAL 30 DAY
                                        THEN q.gsc_clicks ELSE 0 END) AS DOUBLE) /
                        NULLIF(SUM(CASE WHEN q.report_date > b.end_d - INTERVAL 30 DAY
                                        THEN q.gsc_impressions ELSE 0 END), 0), 0) AS ctr_label
            FROM {TABLES['fact_daily']} q
            CROSS JOIN bounds b
            JOIN {TABLES['dim_content']} d ON q.content_hash_id = d.content_hash_id
            WHERE q.report_date > b.end_d - INTERVAL 60 DAY
            GROUP BY q.client_hash_id, q.content_hash_id
            HAVING SUM(CASE WHEN q.report_date > b.end_d - INTERVAL 60 DAY
                            AND q.report_date <= b.end_d - INTERVAL 30 DAY
                           THEN q.gsc_impressions ELSE 0 END) >= 70
        )
        SELECT * FROM windowed
    """).df()
    cache_f.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(cache_f, index=False)
    print("wrote cache:", cache_f)

midline = df["ctr_label"].median()
df["at_risk"] = (df["ctr_label"] < midline).astype(int)
print(f"{len(df):,} content items with enough history")
print("base rate (share at-risk in slice):", f"{df['at_risk'].mean():.3f}")
df.head()

dim_clients                   104 rows
dim_content               519,606 rows
fact_daily_sample      11,694,072 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

wrote cache: /work/outputs/w05_feature_vector.parquet
15,310 content items with enough history
base rate (share at-risk in slice): 0.500


,client_hash_id,content_hash_id,main_intent,content_type,search_volume,impressions_prev30d,impressions_last30d,clicks_30d,avg_position_30d,ctr_label,at_risk
0,client_62f4a7e64f5e0096,content_3478f2c1d077f5a2,commercial,keyword article,10,75.0,1755.0,3.0,12.543007,0.170940,1
1,client_62f4a7e64f5e0096,content_3506ca5554664b6a,transactional,keyword article,10,663.0,122.0,0.0,18.426639,0.000000,1
2,client_62f4a7e64f5e0096,content_35223ef861954493,transactional,keyword article,10,407.0,26219.0,67.0,8.006793,0.255540,0
3,client_62f4a7e64f5e0096,content_35e5e4161f07d648,transactional,keyword article,10,134.0,1545.0,0.0,30.525349,0.000000,1
4,client_62f4a7e64f5e0096,content_364edd6216869946,transactional,keyword article,50,144.0,341.0,1.0,37.901317,0.293255,0



The Week-5 model already used `GroupKFold` by `client_hash_id` (the honest split). This section makes
the choice visible by running the SAME scorers under the naive alternative and showing both numbers.

- **Before — random split.** 5-fold KFold with shuffled rows. The same client can land in both train
  and test, so the model can memorize per-client habits and look better than it is.
- **After — grouped split.** 5-fold GroupKFold on `client_hash_id`. Whole clients are held out, so
  every prediction is for a client the model never trained on — the honest question "does it work on
  a group it never saw?" from the validation skill.

The gap between the two columns is itself the finding: it measures how much of the naive score was
client memorization rather than generalization. Base rate (0.50) is printed as the floor under every
cell.

In [3]:
from sklearn.model_selection import GroupKFold, KFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

CV, SEED = 5, 42
KS = (10, 20, 50)

# honest feature matrix - identical to w05_model (missing flags first, THEN fill)
X = df[["search_volume", "impressions_prev30d", "main_intent", "content_type"]].copy()
X["has_search_volume"] = X["search_volume"].notna().astype(int)
X["search_volume"] = X["search_volume"].fillna(0)
X["main_intent"] = X["main_intent"].fillna("unknown")
X["content_type"] = X["content_type"].fillna("unknown")
X = pd.get_dummies(X, columns=["main_intent", "content_type"], dtype=int)
feature_cols = list(X.columns)

y_risk = df["at_risk"].to_numpy()
groups = df["client_hash_id"].to_numpy()

# same baseline rule as ML-07 / w05, recomputed here
df["visibility_score"] = df["impressions_prev30d"].rank(pct=True)
df["has_sv"] = df["search_volume"].notna()
df["demand_score"] = np.log1p(df["search_volume"].fillna(0)).rank(pct=True) * df["has_sv"]
df["baseline_action_score"] = (
    0.50 * df["visibility_score"]
    + 0.45 * df["demand_score"]
    + 0.05 * (df["main_intent"] == "transactional")
).clip(0, 1)

def precision_at_k(score, y, k):
    order = np.argsort(-np.asarray(score), kind="stable")
    return float(np.asarray(y)[order[:k]].mean())

# the two splits, precomputed so both tables share the same folds
gkf_folds = list(GroupKFold(n_splits=CV).split(X, y_risk, groups))            # AFTER - honest
kf_folds  = list(KFold(n_splits=CV, shuffle=True, random_state=SEED).split(X, y_risk))  # BEFORE - naive
for tr, te in gkf_folds:
    assert not set(groups[tr]) & set(groups[te]), "client overlap between train and test"
print("client overlap across grouped folds: none (holds)")
print("clients in slice:", df["client_hash_id"].nunique())
print("held-out clients per grouped fold:", [len(set(groups[te])) for _, te in gkf_folds])
print("train/test rows per grouped fold:", [(len(t), len(v)) for t, v in gkf_folds])

def evaluate(folds):
    out = {"baseline": [], "lr": []}
    for tr, te in folds:
        out["baseline"].append(df["baseline_action_score"].to_numpy()[te])
        lr = LogisticRegression(max_iter=2000, random_state=SEED)
        lr.fit(X.iloc[tr], y_risk[tr])
        out["lr"].append(lr.predict_proba(X.iloc[te])[:, 1])
    rows = {}
    for name, scores in out.items():
        rows[name] = {k: round(float(np.mean([precision_at_k(scores[i], y_risk[folds[i][1]], k)
                                              for i in range(len(folds))])), 3) for k in KS}

        rows[name]["auc"] = round(float(np.mean([roc_auc_score(y_risk[folds[i][1]], scores[i])
                                                 for i in range(len(folds))])), 3)
    return rows

before = evaluate(kf_folds)    # naive random split
after  = evaluate(gkf_folds)   # honest grouped-by-client split

rows = [["base rate", "-", 0.5, 0.5, 0.5, 0.5]]
for name, label in [("baseline", "rule"), ("lr", "logistic")]:
    for split, d in [("before (random)", before[name]), ("after (grouped)", after[name])]:
        rows.append([label, split, d[10], d[20], d[50], d["auc"]])
table = pd.DataFrame(rows, columns=["scorer", "split", "p@10", "p@20", "p@50", "AUC"])
print()
print(table.to_string(index=False))

client overlap across grouped folds: none (holds)
clients in slice: 14
held-out clients per grouped fold: [1, 1, 1, 1, 10]
train/test rows per grouped fold: [(7575, 7735), (12356, 2954), (12613, 2697), (14293, 1017), (14403, 907)]

   scorer           split  p@10  p@20  p@50   AUC
base rate               -  0.50  0.50 0.500 0.500
     rule before (random)  0.62  0.61 0.572 0.531
     rule after (grouped)  0.60  0.62 0.584 0.513
 logistic before (random)  0.94  0.92 0.908 0.653
 logistic after (grouped)  0.86  0.83 0.844 0.597


### What the before/after comparison says (measured)

- **The model's AUC and p@K drop from random to grouped.** The gap is the size of the memorization:
  in the random split, rows from one client are split across train and test, so the model can copy
  per-client habits; the grouped split forbids that, and the honest number is lower.
- **The rule's cells shift too, but for a different reason.** The rule is not trained, so its
  random-vs-grouped gap is just the two splits putting different rows in test — not a memorization
  signal. The model's gap is the one that carries the finding.
- The base rate row (0.50) is the floor every cell is judged against. The grouped number still clears
  it, which is the honest claim we can make: the model adds skill over the rule even on clients it
  never trained on, just less than the naive number suggested.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

The final feature set is `search_volume`, `impressions_prev30d`, `main_intent`, `content_type` (+
`has_search_volume`). Every one of them exists strictly BEFORE the label window (2025-08-01 -->
2025-08-31): `impressions_prev30d` is days −60…−31, the rest are static attributes known at
creation. The label `at_risk` is `ctr_label < slice median`, where `ctr_label = clicks_30d /
impressions_last30d` lives inside the last 30 days.

Same four demos as the Week-3 hunt, run on this final set, all scored out-of-fold on the grouped folds:

- **A:** feed the label's own numerator (`clicks_30d`) back in --> should collapse toward 1.0.
- **B:** keep the same-window columns (`impressions_last30d`, `avg_position_30d`) --> still too good.
- **C:** honest set --> the real number.
- **D:** grouped vs random on the honest set --> the memorization gap.

In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

def auc_oof(X_use, folds):
    aucs = []
    for tr, te in folds:
        m = RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1)
        m.fit(X_use.iloc[tr], y_risk[tr])
        aucs.append(roc_auc_score(y_risk[te], m.predict_proba(X_use.iloc[te])[:, 1]))
    return float(np.mean(aucs))

# Demo A - leak #1: label-derived feature (the label's own numerator)
leakA = pd.concat([X, df["clicks_30d"]], axis=1)

# Demo B - leak #2: future/overlapping windows (same-window columns, no numerator)
leakB = pd.concat([X, df[["impressions_last30d", "avg_position_30d"]]], axis=1)

# Demo C - honest set: only what exists strictly before the label window
leakC = X

print("base rate (random-ranking floor):", f"{y_risk.mean():.3f}")
for name, Xd in [("A: label numerator (clicks_30d)", leakA),
                 ("B: same-window columns", leakB),
                 ("C: honest set", leakC)]:
    print(f"{name:32} AUC {auc_oof(Xd, gkf_folds):.3f}")

print()
print("Demo D - the memorization gap on the honest set (out-of-fold):")
print(f" ==> random split  AUC {auc_oof(leakC, kf_folds):.3f}")
print(f" ==> grouped split AUC {auc_oof(leakC, gkf_folds):.3f}")

base rate (random-ranking floor): 0.500
A: label numerator (clicks_30d)  AUC 0.854
B: same-window columns           AUC 0.724
C: honest set                    AUC 0.557

Demo D - the memorization gap on the honest set (out-of-fold):
 ==> random split  AUC 0.599
 ==> grouped split AUC 0.557


### Verdict

- **Demo A jumps far above C** — a label-derived feature confesses, exactly as the Week-3 hunt
  predicted. Any score this good is a leak, not skill.
- **Demo B still sits above C** — same-window summaries leak the answer even without the numerator.
- **Demo C is the deployable number**, and **Demo D** shows the random-vs-grouped gap is the
  memorization cost the model paid before.
- Final check: nothing in the feature set touches the label window — features are strictly pre-window
  or static; IDs and product flags are excluded.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Original** (w05's boldest sentence):

> "Logistic Regression wins at every K: 0.86 / 0.83 / 0.844 vs the rule's 0.60 / 0.62 / 0.584,
> against a coin-flip floor of 0.50."

**Why it is too bold:** "wins at every K" reads like a universal verdict, not a measurement of one
slice in one month. Nothing here was randomized, so it cannot support a causal statement about
refreshes; the numbers are per-fold means on held-out clients, not a guarantee about the next client.


**Rewrite (safe, decision-support):**

> Observed on this 15,310-item slice, the logistic regression ranked at-risk pages with higher
> precision@K than the weighted rule across the five held-out-client folds — a directional,
> decision-support result for review prioritization, not a causal claim about refresh outcomes, and
> limited to the slice's 2025-08 window.

In [6]:
original = ("Logistic Regression wins at every K: 0.86 / 0.83 / 0.844 vs the rule's "
            "0.60 / 0.62 / 0.584, against a coin-flip floor of 0.50.")
rewritten = ("Observed on this 15,310-item slice, the logistic regression ranked at-risk pages "
             "with higher precision@K than the weighted rule across the five held-out-client "
             "folds - a directional, decision-support result for review prioritization, not a "
             "causal claim about refresh outcomes, and limited to the slice's 2025-08 window.")
print("ORIGINAL:"); print("  " + original); print()
print("REWRITTEN:"); print("  " + rewritten)

ORIGINAL:
  Logistic Regression wins at every K: 0.86 / 0.83 / 0.844 vs the rule's 0.60 / 0.62 / 0.584, against a coin-flip floor of 0.50.

REWRITTEN:
  Observed on this 15,310-item slice, the logistic regression ranked at-risk pages with higher precision@K than the weighted rule across the five held-out-client folds - a directional, decision-support result for review prioritization, not a causal claim about refresh outcomes, and limited to the slice's 2025-08 window.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.